# SVIX Pipeline: Dummy Data & Architecture Demo
**Author:** Léandre

**Purpose:** To demonstrate how to load the dummy dataset and how the `schemas.py` strict data contracts protect our quantitative pipeline.

In [2]:
import json
import os
import sys
from datetime import datetime

import pandas as pd

# Ensure the src module can be imported from the notebooks folder
sys.path.append(os.path.abspath("../"))
from src.core.schemas import OptionChain

### 1. Loading the Dummy Data
We generated a static volatility surface for testing. The CSV contains the options prices, and the JSON contains the macroeconomic state (\(S, r, q\)).

In [5]:
# Load the surface
df_surface = pd.read_csv("../tests/data/dummy_spx_chain.csv")

# Load the macro state
with open("../tests/data/dummy_spx_context.json", "r") as f:
    macro_state = json.load(f)

print(f"Loaded {len(df_surface)} option records.")
print(f"Macro state: {macro_state}")
display(df_surface.head())

Loaded 64 option records.
Macro state: {'date': '2026-09-25', 'maturity_days': 30, 'underlying_price': 4000.0, 'risk_free_rate': 0.05, 'dividend_yield': 0.01, 'forward_price': 4013.17}


,strike,option_type,bid,ask,mid_price
0,3600.0,C,407.88,416.12,412.00
1,3600.0,P,0.52,0.53,0.53
2,3625.0,C,383.50,391.25,387.38
3,3625.0,P,0.79,0.81,0.80
4,3650.0,C,359.24,366.50,362.87


### 2. Instantiating the Contract (Success)
Here we package the raw data into our immutable `OptionChain` contract. If the data is mathematically sound, the instantiation succeeds.

In [6]:
valid_chain = OptionChain(
    date=datetime.strptime(macro_state["date"], "%Y-%m-%d"),
    maturity_days=macro_state["maturity_days"],
    underlying_price=macro_state["underlying_price"],
    risk_free_rate=macro_state["risk_free_rate"],
    forward_price=macro_state["forward_price"],
    dividend_yield=macro_state["dividend_yield"],
    options_data=df_surface,
)

print("SUCCESS: OptionChain instantiated perfectly.")
print(f"Forward price locked at: {valid_chain.forward_price}")

SUCCESS: OptionChain instantiated perfectly.
Forward price locked at: 4013.17


### 3. Demonstrating the Guardrails (Failure)
What happens if the data engineering pipeline accidentally passes corrupted data (e.g., negative maturity, or missing columns)? The contract will actively block it.

In [7]:
try:
    invalid_chain = OptionChain(
        date=datetime.strptime(macro_state["date"], "%Y-%m-%d"),
        maturity_days=-5,  # <-- MALICIOUS DATA INJECTED HERE
        underlying_price=macro_state["underlying_price"],
        risk_free_rate=macro_state["risk_free_rate"],
        forward_price=macro_state["forward_price"],
        dividend_yield=macro_state["dividend_yield"],
        options_data=df_surface,
    )
except ValueError as e:
    print(f"GUARDRAIL TRIGGERED: {e}")

GUARDRAIL TRIGGERED: Invalid maturity: -5 days.
